In [ ]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

csv_path = r"data\unified_bio_discussion_data.csv"
output_path = r"data\clustered_bio_innovations.csv"

# Load  corpus
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} posts for topic modeling.")

docs = df['processed_text'].dropna().tolist()


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


vectorizer_model = CountVectorizer(stop_words="english", min_df=2)

# BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True
)

# Fit BERTopic
print("Fitting BERTopic model (this may take a moment)...")
topics, probs = topic_model.fit_transform(docs)

topic_info = topic_model.get_topic_info()
print("\n=== DISCOVERED BIO-BASED INNOVATION TOPICS ===")
display(topic_info[['Topic', 'Count', 'Name']].head(10)) 
df['assigned_topic'] = topics
df['topic_representation'] = df['assigned_topic'].map(
    lambda t: " ".join([word for word, _ in topic_model.get_topic(t)[:4]]) if t != -1 else "Outlier/General"
)
df.to_csv(output_path, index=False)
print(f"\n[✔] Clustering complete. Results saved to: {output_path}")

Loaded 100 posts for topic modeling.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

2026-08-08 13:41:28,000 - BERTopic - Embedding - Completed ✓
2026-08-08 13:41:28,001 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-08 13:41:43,221 - BERTopic - Dimensionality - Completed ✓
2026-08-08 13:41:43,223 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-08 13:41:43,250 - BERTopic - Cluster - Completed ✓
2026-08-08 13:41:43,252 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-08 13:41:43,287 - BERTopic - Representation - Completed ✓
2026-08-08 13:41:43,290 - BERTopic - Topic reduction - Reducing number of topics
2026-08-08 13:41:43,303 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-08 13:41:43,325 - BERTopic - Representation - Completed ✓
2026-08-08 13:41:43,328 - BERTopic - Topic reduction - Reduced number of topics from 2 to 2



=== DISCOVERED BIO-BASED INNOVATION TOPICS ===


,Topic,Count,Name
0,0,75,0_wood_new_don_know
1,1,25,1_just_use_structural_risk



[✔] Clustering complete. Results saved to: data\clustered_bio_innovations.csv


In [17]:
"""
Explainable Behavioural Bias Detection in Consumer Discussions on Bio-Based Innovations
Full Pipeline: Zero-shot Classification + BERTopic + KeyBERT + Statistical Testing
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


# IMPORTS


import transformers
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from scipy.stats import mannwhitneyu

#  KeyBERT
try:
    from keybert import KeyBERT
    KEYBERT_AVAILABLE = True
except ImportError:
    KEYBERT_AVAILABLE = False
    print("KeyBERT not installed. Install with: pip install keybert")


# CONFIGURATION


#  clustered data
CSV_PATH = r"data\clustered_bio_innovations.csv"
OUTPUT_PATH = r"data\bias_analysis_results.csv"
SUMMARY_PATH = r"data\bias_summary.csv"


# NATURAL LANGUAGE HYPOTHESES (Better zero-shot labels)


HYPOTHESIS_LABELS = [
    "The consumer prefers existing products over new alternatives.",
    "The consumer avoids this product because its performance is uncertain.",
    "The consumer fears potential losses from adopting this product.",
    "The consumer trusts products they already know and use.",
    "The consumer does not believe the product will perform as claimed.",
    "The consumer believes others in their community do not use this product."
]

DISPLAY_LABELS = [
    "status quo bias",
    "ambiguity aversion",
    "loss aversion",
    "familiarity bias",
    "trust",
    "social norms"
]

LABEL_MAP = dict(zip(DISPLAY_LABELS, HYPOTHESIS_LABELS))
DISPLAY_MAP = dict(zip(HYPOTHESIS_LABELS, DISPLAY_LABELS))


# HELPER FUNCTION 

def safe_bar(value):
    """Create a bar string safely handling NaN values."""
    if pd.isna(value) or np.isnan(value):
        return " " * 20
    val = int(value)
    return "█" * val + "░" * (20 - val) if val <= 20 else "█" * 20


# LOAD DATA


print("=" * 70)
print(" EXPLAINABLE BEHAVIOURAL BIAS DETECTION SYSTEM")
print(" Methodological Contribution: An explainable NLP framework for")
print(" identifying behavioural barriers to bio-based innovation adoption.")
print("=" * 70)

print("\n[1] Loading data...")
df = pd.read_csv(CSV_PATH)
print(f"    Loaded {len(df)} texts.")



print("\n[1b] Standardising language labels...")

# Check  column 
if 'language' in df.columns:
   
    original_langs = df['language'].unique()
    print(f"    Original language values found: {original_langs}")
    
    #  uppercase
    df['language'] = (
        df['language']
        .astype(str)
        .str.strip()
        .str.upper()
    )
    
    # Map variations to standard codes
    df['language'] = df['language'].replace({
        'ENGLISH': 'EN',
        'ENG': 'EN',
        'E': 'EN',
        'FINNISH': 'FI',
        'SUOMI': 'FI',
        'FIN': 'FI',
        'F': 'FI'
    })
    
    #  cases where source column indicate language
    if 'source' in df.columns:
        df.loc[df['source'].str.upper().str.contains('SUOMI'), 'language'] = 'FI'
        df.loc[df['source'].str.upper().str.contains('REDDIT'), 'language'] = 'EN'
    
    #  check
    lang_counts = df['language'].value_counts()
    print(f"    Standardised language distribution:")
    for lang, count in lang_counts.items():
        print(f"      {lang}: {count} texts")
    
    #  Language Sepration
    lang_en = df[df['language'].isin(['EN', 'ENGLISH'])]
    lang_fi = df[df['language'].isin(['FI', 'FINNISH', 'SUOMI'])]
    
    print(f"    English texts: {len(lang_en)}")
    print(f"    Finnish texts: {len(lang_fi)}")
else:
    print("    No 'language' column found. Language analysis will be skipped.")
    lang_en = pd.DataFrame()
    lang_fi = pd.DataFrame()

#  processed_text (already translated to English)
texts = df['processed_text'].dropna().tolist()
print(f"\n    {len(texts)} texts available for analysis.")


# INITIALIZE CLASSIFIER


print("\n[2] Loading zero-shot classifier...")
print("    Model: facebook/bart-large-mnli")
print("    (This may take 1-2 minutes to download the model)")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    return_all_scores=True,
    device=0,
    batch_size=8
)

print(f"    Model loaded successfully.")
print(f"    Target biases: {', '.join(DISPLAY_LABELS)}")
print("\n    Using natural language hypotheses:")
for i, label in enumerate(DISPLAY_LABELS):
    print(f"      {i+1}. {label}: {HYPOTHESIS_LABELS[i]}")


# BATCH PREDICTION FUNCTION


def predict_bias_probabilities(texts_batch):
    """Predict bias probabilities for a batch of texts."""
    if isinstance(texts_batch, str):
        texts_batch = [texts_batch]
    elif isinstance(texts_batch, np.ndarray):
        texts_batch = texts_batch.tolist()
    
    if not texts_batch:
        return np.array([])
    
    results = classifier(texts_batch, HYPOTHESIS_LABELS, multi_label=True)
    
    if isinstance(results, dict):
        results = [results]
    
    probs = []
    for res in results:
        label_scores = {label: score for label, score in zip(res['labels'], res['scores'])}
        probs.append([label_scores[label] for label in HYPOTHESIS_LABELS])
    
    return np.array(probs)


#  FULL CLASSIFICATION


print("\n[3] Classifying all texts for bias detection...")
print("    Processing in batches...")

batch_size = 16
all_probs = []
total_batches = (len(texts) + batch_size - 1) // batch_size

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    probs = predict_bias_probabilities(batch)
    all_probs.append(probs)
    print(f"    Processed batch {i//batch_size + 1}/{total_batches}")

probs_array = np.vstack(all_probs)
print(f"    Classification complete for {len(texts)} texts.")

# AGGREGATE ANALYSIS


print("\n[4] Calculating aggregate statistics...")

bias_cols = [label.replace(" ", "_").lower() for label in DISPLAY_LABELS]
for i, col in enumerate(bias_cols):
    df[col] = probs_array[:, i]

mean_probs = probs_array.mean(axis=0)
std_probs = probs_array.std(axis=0)

print("\n" + "=" * 70)
print(" AGGREGATE BIAS PREVALENCE REPORT")
print("=" * 70)
print(f"\nOverall bias prevalence across all texts (n = {len(texts)}):")
print("-" * 50)

for i, label in enumerate(DISPLAY_LABELS):
    mean = mean_probs[i] * 100
    std = std_probs[i] * 100
    bar = safe_bar(mean / 5)
    print(f"  {label.title():<20}: {mean:5.1f}%  (± {std:.1f}%)  {bar}")

dominant_bias_indices = np.argmax(probs_array, axis=1)
dominant_counts = pd.Series(dominant_bias_indices).value_counts()

print("\n" + "-" * 50)
print("Dominant bias distribution:")
for idx, count in dominant_counts.items():
    label = DISPLAY_LABELS[idx]
    percentage = (count / len(texts)) * 100
    print(f"  {label.title():<20}: {count:>3} texts ({percentage:.1f}%)")


# TOPIC-BIAS MAPPING


print("\n[5] Analysing bias distribution by topic...")

TOPIC_NAMES = {
    0: "Uncertainty About Novel Bio-Materials",
    1: "Risk Aversion in Material Decisions",
    -1: "Outlier/General Discussions"
}

if 'assigned_topic' in df.columns:
    print("\nBias prevalence by BERTopic cluster:")
    print("-" * 70)
    
    topic_groups = df.groupby('assigned_topic')
    for topic, group in topic_groups:
        topic_name = TOPIC_NAMES.get(topic, f"Topic {topic}")
        rep = group['topic_representation'].iloc[0] if 'topic_representation' in df.columns else ""
        
        print(f"\n  Topic {topic}: {topic_name}")
        print(f"  {len(group)} texts")
        print(f"  Keywords: {rep}")
        
        for i, label in enumerate(DISPLAY_LABELS):
            col = bias_cols[i]
            mean = group[col].mean()
            if pd.isna(mean):
                mean = 0.0
            mean_pct = mean * 100
            bar = safe_bar(mean_pct / 5)
            print(f"    {label.title():<20}: {mean_pct:.1f}%  {bar}")
        
        topic_probs = group[bias_cols].values
        dom_idx = np.argmax(np.mean(topic_probs, axis=0))
        print(f"    Dominant bias: {DISPLAY_LABELS[dom_idx].title()}")




print("\n[6] Analysing bias differences by language...")

missing_cols = [col for col in bias_cols if col not in df.columns]
if missing_cols:
    print(f"    WARNING: The following bias columns are missing: {missing_cols}")
    print("    Attempting to add them with default values...")
    for col in missing_cols:
        df[col] = 0.0

# Check  language data
if 'language' in df.columns:
    # Standardise language labels
    df['language'] = (
        df['language']
        .astype(str)
        .str.strip()
        .str.upper()
        .replace({
            'ENGLISH': 'EN',
            'ENG': 'EN',
            'E': 'EN',
            'FINNISH': 'FI',
            'SUOMI': 'FI',
            'FIN': 'FI',
            'F': 'FI'
        })
    )
    
    
    if 'source' in df.columns:
        df.loc[df['source'].str.upper().str.contains('SUOMI'), 'language'] = 'FI'
        df.loc[df['source'].str.upper().str.contains('REDDIT'), 'language'] = 'EN'
    
    lang_en = df[df['language'] == 'EN']
    lang_fi = df[df['language'] == 'FI']
    
    if len(lang_en) > 0 and len(lang_fi) > 0:
        print(f"\n  ENGLISH: {len(lang_en)} texts")
        for i, label in enumerate(DISPLAY_LABELS):
            col = bias_cols[i]
            if col in lang_en.columns:
                mean = lang_en[col].mean()
                if pd.isna(mean):
                    mean = 0.0
                mean_pct = mean * 100
                bar = safe_bar(mean_pct / 5)
                print(f"    {label.title():<20}: {mean_pct:.1f}%  {bar}")
            else:
                print(f"    {label.title():<20}: Column not found")
        
        print(f"\n  FINNISH: {len(lang_fi)} texts")
        for i, label in enumerate(DISPLAY_LABELS):
            col = bias_cols[i]
            if col in lang_fi.columns:
                mean = lang_fi[col].mean()
                if pd.isna(mean):
                    mean = 0.0
                mean_pct = mean * 100
                bar = safe_bar(mean_pct / 5)
                print(f"    {label.title():<20}: {mean_pct:.1f}%  {bar}")
            else:
                print(f"    {label.title():<20}: Column not found")
    else:
        print("\n  No data available for one or both language groups.")
        print(f"  English: {len(lang_en)} texts, Finnish: {len(lang_fi)} texts")
else:
    print("\n  No 'language' column found. Skipping language analysis.")


# STATISTICAL TESTING 

print("\n[6b] Statistical testing for language differences...")
print("-" * 70)

if 'language' in df.columns and len(lang_en) > 0 and len(lang_fi) > 0:
    print("\nMann-Whitney U Test (English vs Finnish):")
    
    for i, label in enumerate(DISPLAY_LABELS):
        col = bias_cols[i]
        if col not in df.columns:
            print(f"  {label.title():<20}: Column missing, skipping")
            continue
        
        en_vals = lang_en[col].dropna().values
        fi_vals = lang_fi[col].dropna().values
        
        if len(en_vals) > 0 and len(fi_vals) > 0:
            try:
                stat, p_value = mannwhitneyu(en_vals, fi_vals, alternative='two-sided')
                significant = p_value < 0.05
                sig_indicator = "✅ SIGNIFICANT" if significant else "❌ NOT SIGNIFICANT"
                print(f"  {label.title():<20}: p = {p_value:.4f}  {sig_indicator}")
            except Exception as e:
                print(f"  {label.title():<20}: Test failed - {str(e)[:30]}")
        else:
            print(f"  {label.title():<20}: Insufficient data")
else:
    print("\n  Language data not available for statistical testing.")
    print("  (This may be because the dataset is predominantly one language)")

print("\n" + "-" * 70)

# KEYBERT EXPLANATION 

print("\n[7] Generating interpretable explanations with KeyBERT...")

if KEYBERT_AVAILABLE:
    kw_model = KeyBERT()
    print("    KeyBERT loaded successfully.")
else:
    print("    KeyBERT not available. Installing...")
    import subprocess
    subprocess.check_call(["pip", "install", "keybert"])
    from keybert import KeyBERT
    kw_model = KeyBERT()
    print("    KeyBERT installed and loaded.")

# Select sample texts
sample_indices = []
if 'assigned_topic' in df.columns:
    for topic in df['assigned_topic'].unique():
        topic_df = df[df['assigned_topic'] == topic]
        if len(topic_df) > 0:
            sample_idx = topic_df.iloc[len(topic_df)//2].name
            sample_indices.append(sample_idx)
else:
    sample_indices = list(range(0, min(len(texts), 20), 2))

sample_indices = sample_indices[:10]
sample_texts = [texts[idx] for idx in sample_indices if idx < len(texts)]

print("\n" + "=" * 70)
print(" KEYWORD EVIDENCE FOR BIAS CLASSIFICATION")
print(" (KeyBERT attribution - evidence supporting classification)")
print("=" * 70)

for i, text in enumerate(sample_texts):
    print(f"\n{'='*70}")
    print(f"SAMPLE TEXT {i+1}:")
    print(f"'{text}'")
    
    base_probs = predict_bias_probabilities([text])[0]
    
    print("\nBIAS PROBABILITIES:")
    for idx, label in enumerate(DISPLAY_LABELS):
        percentage = base_probs[idx] * 100
        bar = safe_bar(percentage / 5)
        print(f"  {label.title():<20}: {percentage:5.1f}%  {bar}")
    
    dom_idx = np.argmax(base_probs)
    dom_bias = DISPLAY_LABELS[dom_idx]
    dom_conf = base_probs[dom_idx] * 100
    
    print(f"\n  DOMINANT BIAS: {dom_bias.title()} ({dom_conf:.1f}%)")
    
    #  Added explanatory sentence 
    print("\n  The following keyphrases are semantically representative of the")
    print("  discussion and provide qualitative evidence supporting the")
    print("  predicted behavioural bias:")
    
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words='english',
        top_n=5
    )
    
    if keywords:
        for word, score in keywords:
            print(f"    • '{word}' (relevance: {score:.3f})")
    
    print("-" * 40)


# BIAS-KEYWORD MAPPING


print("\n" + "=" * 70)
print(" BIAS-KEYWORD EVIDENCE MAPPING")
print(" (Aggregate evidence patterns across the dataset)")
print("=" * 70)

bias_keyword_map = {label: [] for label in DISPLAY_LABELS}

print("\nExtracting evidence patterns for each bias category...")

sample_df = df.sample(min(50, len(df)))

for idx, row in sample_df.iterrows():
    text = row['processed_text']
    probs = predict_bias_probabilities([text])[0]
    dom_idx = np.argmax(probs)
    dom_bias = DISPLAY_LABELS[dom_idx]
    
    try:
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=3
        )
        for word, score in keywords:
            if len(bias_keyword_map[dom_bias]) < 10:
                if word not in [w for w, _ in bias_keyword_map[dom_bias]]:
                    bias_keyword_map[dom_bias].append((word, score))
    except:
        continue

print("\nKeywords associated with each bias (evidence patterns):")
print("-" * 50)
for bias, keywords in bias_keyword_map.items():
    if keywords:
        sorted_kw = sorted(keywords, key=lambda x: x[1], reverse=True)[:5]
        words = [w for w, _ in sorted_kw]
        print(f"  {bias.title():<20}: {', '.join(words)}")
    else:
        print(f"  {bias.title():<20}: (no evidence extracted)")


# COMMUNICATION RECOMMENDATIONS


print("\n" + "=" * 70)
print(" COMMUNICATION RECOMMENDATIONS")
print(" (Translating behavioural insights into actionable strategies)")
print("=" * 70)

recommendations = {
    "loss_aversion": """
      • Provide performance guarantees and warranties
      • Offer insurance options to reduce perceived financial risk
      • Share case studies of successful adoption with cost-benefit analysis
      • Highlight long-term savings despite upfront costs""",
    
    "ambiguity_aversion": """
      • Publish long-term durability data and third-party certifications
      • Address unknowns directly with clear, accessible documentation
      • Share transparent testing results from independent labs
      • Create decision-support tools that quantify performance uncertainty""",
    
    "status_quo_bias": """
      • Highlight incremental adoption pathways
      • Show integration into existing workflows with minimal disruption
      • Feature testimonials from early adopters
      • Demonstrate compatibility with current practices""",
    
    "familiarity_bias": """
      • Create demonstration projects and pilot programs
      • Offer hands-on training to increase familiarity
      • Provide samples and trial periods
      • Partner with trusted industry figures as ambassadors""",
    
    "trust": """
      • Partner with trusted third-party certifiers
      • Be transparent about material limitations
      • Provide clear, verifiable performance metrics
      • Engage with community feedback openly""",
    
    "social_norms": """
      • Showcase growing adoption rates
      • Frame bio-based adoption as a progressive, forward-thinking choice
      • Highlight peer adoption stories
      • Share community success examples"""
}

mean_probs_sorted = sorted(
    [(DISPLAY_LABELS[i], mean_probs[i]) for i in range(len(DISPLAY_LABELS))],
    key=lambda x: x[1],
    reverse=True
)

print("\nBased on the aggregate bias analysis, the following communication")
print("strategies are recommended for promoting bio-based innovations:\n")

for bias, prob in mean_probs_sorted[:3]:
    if prob > 0.3:
        key = bias.replace(" ", "_").lower()
        print(f"\n{'='*50}")
        print(f"  {bias.upper()} ({prob*100:.1f}% prevalence)")
        print("=" * 50)
        if key in recommendations:
            print(recommendations[key])


# METHODOLOGICAL CONTRIBUTION


print("\n" + "=" * 70)
print(" METHODOLOGICAL CONTRIBUTION")
print("=" * 70)
print("""
This pilot study presents an explainable NLP framework for identifying 
behavioural barriers to bio-based innovation adoption. By combining:
  1. Zero-shot classification for bias detection
  2. BERTopic for thematic clustering
  3. KeyBERT for interpretable evidence attribution
  4. Statistical testing for validation

The framework provides a scalable, data-driven approach to translating 
consumer insights into actionable communication strategies for firms and 
policymakers in the forest sector.

This methodological contribution bridges:
  • Computational Social Science
  • Behavioural Economics
  • Sustainability Communication
  • Explainable AI
""")


#  RESULTS


print("\n[8] Saving results...")

df.to_csv(OUTPUT_PATH, index=False)
print(f"    Results saved to: {OUTPUT_PATH}")

summary_data = []
for i, label in enumerate(DISPLAY_LABELS):
    summary_data.append({
        'bias': label,
        'mean_probability': mean_probs[i] * 100,
        'std_probability': std_probs[i] * 100
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"    Summary saved to: {SUMMARY_PATH}")


# FINAL OUTPUT


print("\n" + "=" * 70)
print(" ANALYSIS COMPLETE")
print("=" * 70)

print("""
The following outputs have been generated:
  1. Aggregate bias prevalence statistics
  2. Bias distribution by BERTopic cluster (with improved topic names)
  3. Statistical testing for language differences (Mann-Whitney U)
  4. KeyBERT evidence attribution for sample texts
  5. Bias-keyword evidence mapping
  6. Communication recommendations for firms and policymakers
  7. Full results saved to: data/bias_analysis_results.csv
  8. Summary saved to: data/bias_summary.csv


""")

 EXPLAINABLE BEHAVIOURAL BIAS DETECTION SYSTEM
 Methodological Contribution: An explainable NLP framework for
 identifying behavioural barriers to bio-based innovation adoption.

[1] Loading data...
    Loaded 100 texts.

[1b] Standardising language labels...
    Original language values found: ['en' 'fi']
    Standardised language distribution:
      EN: 75 texts
      FI: 25 texts
    English texts: 75
    Finnish texts: 25

    100 texts available for analysis.

[2] Loading zero-shot classifier...
    Model: facebook/bart-large-mnli
    (This may take 1-2 minutes to download the model)


Device set to use cuda:0


    Model loaded successfully.
    Target biases: status quo bias, ambiguity aversion, loss aversion, familiarity bias, trust, social norms

    Using natural language hypotheses:
      1. status quo bias: The consumer prefers existing products over new alternatives.
      2. ambiguity aversion: The consumer avoids this product because its performance is uncertain.
      3. loss aversion: The consumer fears potential losses from adopting this product.
      4. familiarity bias: The consumer trusts products they already know and use.
      5. trust: The consumer does not believe the product will perform as claimed.
      6. social norms: The consumer believes others in their community do not use this product.

[3] Classifying all texts for bias detection...
    Processing in batches...
    Processed batch 1/7
    Processed batch 2/7
    Processed batch 3/7
    Processed batch 4/7
    Processed batch 5/7
    Processed batch 6/7
    Processed batch 7/7
    Classification complete for 100 